# 🔬 Task 5: Model Evaluation with Ground Truth

This section demonstrates how to evaluate the model against **ground truth answers**.

We use the **VQA-RAD dataset** - a radiology VQA benchmark with expert-annotated answers.

---

In [ ]:
# ============================================================================
# EVALUATION WITH GROUND TRUTH - VQA-RAD DATASET
# ============================================================================
# VQA-RAD is a benchmark dataset for Visual Question Answering in Radiology
# containing 315 images with 3,515 question-answer pairs annotated by clinicians.
#
# Reference: Lau et al., "A Dataset for Visual Question Answering in Radiology"
# https://www.nature.com/articles/sdata2018251
# ============================================================================

# Since VQA-RAD requires registration, we'll create a sample evaluation set
# with realistic medical imaging Q&A pairs that simulate ground truth evaluation

# Sample evaluation data (simulating VQA-RAD format)
# In production, you would load the actual VQA-RAD dataset
EVALUATION_SET = [
    {
        "image_idx": 0,  # Use our downloaded images
        "question": "Is this a chest X-ray?",
        "ground_truth": "yes",
        "answer_type": "closed"  # yes/no question
    },
    {
        "image_idx": 0,
        "question": "What imaging modality is shown?",
        "ground_truth": "x-ray",
        "answer_type": "closed"
    },
    {
        "image_idx": 0,
        "question": "What body part is shown in this image?",
        "ground_truth": "chest",
        "answer_type": "closed"
    },
    {
        "image_idx": 0,
        "question": "Is the heart visible in this image?",
        "ground_truth": "yes",
        "answer_type": "closed"
    },
    {
        "image_idx": 0,
        "question": "Are the lungs visible?",
        "ground_truth": "yes",
        "answer_type": "closed"
    },
]

print(f"📊 Evaluation set: {len(EVALUATION_SET)} questions")
print("   Question types: closed-ended (yes/no, one-word answers)")

In [ ]:
# ============================================================================
# EVALUATION METRICS
# ============================================================================
# For Medical VQA, we use:
# 1. Exact Match Accuracy - strict matching
# 2. Relaxed Accuracy - checks if ground truth is contained in response
# 3. BLEU/ROUGE - for open-ended questions (optional)
# ============================================================================

import re

def normalize_answer(text):
    """Normalize answer for comparison."""
    text = text.lower().strip()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Common medical abbreviations
    text = text.replace('x-ray', 'xray').replace('x ray', 'xray')
    text = text.replace('ct scan', 'ct').replace('mri scan', 'mri')
    return text

def evaluate_answer(prediction, ground_truth, answer_type="closed"):
    """
    Evaluate model prediction against ground truth.
    
    Returns:
        dict with 'exact_match' and 'relaxed_match' scores
    """
    pred_norm = normalize_answer(prediction)
    gt_norm = normalize_answer(ground_truth)
    
    # Exact match
    exact_match = gt_norm in pred_norm.split()[:10]  # Check first 10 words
    
    # Relaxed match (GT appears anywhere in response)
    relaxed_match = gt_norm in pred_norm
    
    # For yes/no questions, check for affirmative/negative
    if answer_type == "closed" and ground_truth.lower() in ["yes", "no"]:
        yes_indicators = ["yes", "correct", "affirmative", "indeed", "is a", "are visible", "is visible", "shows"]
        no_indicators = ["no", "not", "incorrect", "negative", "absent", "cannot"]
        
        if ground_truth.lower() == "yes":
            relaxed_match = any(ind in pred_norm for ind in yes_indicators)
        else:
            relaxed_match = any(ind in pred_norm for ind in no_indicators)
    
    return {
        "exact_match": exact_match,
        "relaxed_match": relaxed_match
    }

print("✅ Evaluation functions defined")

In [ ]:
# ============================================================================
# RUN EVALUATION
# ============================================================================

def run_evaluation(eval_set, images, max_samples=5):
    """
    Run model evaluation on a set of Q&A pairs with ground truth.
    """
    results = []
    
    print("\n" + "=" * 70)
    print("🔬 RUNNING GROUND TRUTH EVALUATION")
    print("=" * 70)
    
    for i, sample in enumerate(eval_set[:max_samples]):
        print(f"\n📝 Question {i+1}/{min(len(eval_set), max_samples)}")
        print(f"   Q: {sample['question']}")
        print(f"   Ground Truth: {sample['ground_truth']}")
        
        # Get model prediction
        img_path = images[sample['image_idx']]["path"]
        pred_result = diagnose_xray(img_path, sample['question'], max_new_tokens=50)
        prediction = pred_result['response']
        
        # Evaluate
        eval_result = evaluate_answer(prediction, sample['ground_truth'], sample['answer_type'])
        
        print(f"   Model Answer: {prediction[:100]}..." if len(prediction) > 100 else f"   Model Answer: {prediction}")
        print(f"   ✅ Relaxed Match: {eval_result['relaxed_match']}")
        
        results.append({
            'question': sample['question'],
            'ground_truth': sample['ground_truth'],
            'prediction': prediction,
            'exact_match': eval_result['exact_match'],
            'relaxed_match': eval_result['relaxed_match'],
            'ttft': pred_result['ttft']
        })
    
    # Calculate overall metrics
    exact_acc = sum(r['exact_match'] for r in results) / len(results) * 100
    relaxed_acc = sum(r['relaxed_match'] for r in results) / len(results) * 100
    avg_ttft = sum(r['ttft'] for r in results) / len(results)
    
    print("\n" + "=" * 70)
    print("📊 EVALUATION RESULTS")
    print("=" * 70)
    print(f"\n   Total Questions:     {len(results)}")
    print(f"   Exact Match Accuracy: {exact_acc:.1f}%")
    print(f"   Relaxed Accuracy:     {relaxed_acc:.1f}%")
    print(f"   Average TTFT:         {avg_ttft:.3f}s")
    
    return results, {'exact_acc': exact_acc, 'relaxed_acc': relaxed_acc, 'avg_ttft': avg_ttft}

# Run the evaluation
if downloaded_images:
    eval_results, metrics = run_evaluation(EVALUATION_SET, downloaded_images)
else:
    print("⚠️ No images available for evaluation")

In [ ]:
# ============================================================================
# VISUALIZE EVALUATION RESULTS
# ============================================================================

def visualize_evaluation(results, metrics):
    """Create a visualization of evaluation results."""
    plt.style.use('dark_background')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#1a1a2e')
    
    # Chart 1: Accuracy comparison
    ax1 = axes[0]
    accuracies = [metrics['exact_acc'], metrics['relaxed_acc']]
    labels = ['Exact Match', 'Relaxed Match']
    colors = ['#ff6b6b', '#00ff88']
    
    bars = ax1.bar(labels, accuracies, color=colors, edgecolor='white', linewidth=2)
    for bar, val in zip(bars, accuracies):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                 f'{val:.1f}%', ha='center', fontsize=14, fontweight='bold', color='white')
    
    ax1.set_ylim(0, 110)
    ax1.set_ylabel('Accuracy (%)', fontsize=12, color='white')
    ax1.set_title('📊 Model Accuracy vs Ground Truth', fontsize=14, fontweight='bold', color='#00d4ff')
    ax1.set_facecolor('#16213e')
    ax1.tick_params(colors='white')
    ax1.axhline(y=80, color='#ffd93d', linestyle='--', alpha=0.5, label='Clinical Threshold (80%)')
    ax1.legend(loc='lower right', facecolor='#16213e', edgecolor='white')
    
    # Chart 2: Per-question results
    ax2 = axes[1]
    questions = [f"Q{i+1}" for i in range(len(results))]
    matches = [r['relaxed_match'] for r in results]
    bar_colors = ['#00ff88' if m else '#ff6b6b' for m in matches]
    
    ax2.bar(questions, [1]*len(questions), color=bar_colors, edgecolor='white', linewidth=2)
    ax2.set_ylim(0, 1.2)
    ax2.set_title('📝 Per-Question Results', fontsize=14, fontweight='bold', color='#00d4ff')
    ax2.set_facecolor('#16213e')
    ax2.tick_params(colors='white')
    ax2.set_ylabel('Correct (green) / Incorrect (red)', fontsize=10, color='white')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#00ff88', label='Correct'),
                       Patch(facecolor='#ff6b6b', label='Incorrect')]
    ax2.legend(handles=legend_elements, loc='upper right', facecolor='#16213e', edgecolor='white')
    
    fig.suptitle('🔬 Ground Truth Evaluation Results', fontsize=16, fontweight='bold', color='white', y=1.02)
    plt.tight_layout()
    plt.savefig('evaluation_results.png', dpi=150, facecolor='#1a1a2e', bbox_inches='tight')
    print("💾 Saved: evaluation_results.png")
    plt.show()

if 'eval_results' in dir() and 'metrics' in dir():
    visualize_evaluation(eval_results, metrics)

---
## 📚 How to Use Real Evaluation Datasets

For your PhD interview, you can mention these **standard Medical VQA benchmarks**:

| Dataset | Size | Modality | Access |
|---------|------|----------|--------|
| **VQA-RAD** | 3,515 Q&A | Radiology | [Request access](https://osf.io/89kps/) |
| **PathVQA** | 32,799 Q&A | Pathology | [HuggingFace](https://huggingface.co/datasets/flaviagiammarino/path-vqa) |
| **SLAKE** | 14,028 Q&A | Multi-modal | [GitHub](https://github.com/Slake-dataset/SLAKE) |
| **VQA-Med** | 15,292 Q&A | Radiology | [ImageCLEF](https://www.imageclef.org/2021/medical/vqa) |

### Loading PathVQA (Example):
```python
from datasets import load_dataset
pathvqa = load_dataset("flaviagiammarino/path-vqa", split="test")

for sample in pathvqa:
    image = sample['image']
    question = sample['question']
    ground_truth = sample['answer']
    # Evaluate your model here
```

### Key Metrics to Report:
- **Closed-ended accuracy**: For yes/no questions
- **Open-ended accuracy**: For descriptive answers (use BLEU/ROUGE)
- **Per-category breakdown**: Modality, organ, question type

In [ ]:
# ============================================================================
# OPTIONAL: LOAD PATHVQA FOR REAL EVALUATION
# ============================================================================
# Uncomment to run on actual benchmark dataset
# ============================================================================

# !pip install -q datasets

# from datasets import load_dataset
# 
# print("Loading PathVQA test set...")
# pathvqa = load_dataset("flaviagiammarino/path-vqa", split="test[:20]")  # First 20 samples
# 
# real_results = []
# for i, sample in enumerate(pathvqa):
#     print(f"Evaluating {i+1}/20...")
#     
#     # Save image temporarily
#     img_path = f"temp_eval_{i}.png"
#     sample['image'].save(img_path)
#     
#     # Get prediction
#     result = diagnose_xray(img_path, sample['question'], max_new_tokens=30)
#     
#     # Evaluate
#     eval_score = evaluate_answer(result['response'], sample['answer'])
#     real_results.append(eval_score['relaxed_match'])
# 
# print(f"\nPathVQA Accuracy: {sum(real_results)/len(real_results)*100:.1f}%")